# Data cleaning

This notebook converts nested Shopify exports into analysis-ready tables with explicit grains, keys, and relationships. Raw inputs remain unchanged; all transformations occur in working copies before clean tables are created.

The central rule is: **determine what one row represents before removing repeated IDs.** A repeated ID may indicate a legitimate child relationship rather than a bad duplicate.

## 1. Load and preserve source data

Load the newest dated export from each configured folder. The `*_raw` tables preserve the source exactly as imported, while later `*_work` tables receive normalized IDs and investigation fields.

In [ ]:
from pathlib import Path
import os

import pandas as pd


# Local data folder. Set this in your shell before launching Jupyter or VS Code.
data_dir = os.environ.get('RETAIL_DATA_DIR')
if not data_dir:
    raise EnvironmentError('Set RETAIL_DATA_DIR to your local Retail Business data folder before running this notebook.')
DATA_DIR = Path(data_dir).expanduser()


def latest_file(folder, suffix):
    """Return the latest ISO-dated export matching the expected suffix."""
    matches = sorted(folder.glob(f"*{suffix}"))
    if not matches:
        raise FileNotFoundError(f"No files matching *{suffix} in {folder}")
    return matches[-1]


folders = {
    "customers": DATA_DIR / "Dim_Customers",
    "orders": DATA_DIR / "Fact_Orders",
    "products": DATA_DIR / "Dim_Products",
    "inventory": DATA_DIR / "Dim_Inventory",
    "variants": DATA_DIR / "Dim_Variants"
}

suffixes = {
    "customers": "_Customers.csv",
    "orders": "_Orders.csv",
    "products": "_Products.csv",
    "inventory": "_Inventory_Items.csv",
    "variants": "_Variants.csv"
}

tables = {}
source_files = {}

for name, folder in folders.items():
    source_file = latest_file(folder, suffixes[name])
    source_files[name] = source_file
    tables[name] = pd.read_csv(source_file)
    print(f"{name}: {source_file.name} — {len(tables[name]):,} raw rows")

In [ ]:
# Preserve untouched source tables. Cleaning will produce separate *_clean tables.
customers_raw = tables["customers"].copy()
orders_raw = tables["orders"].copy()
products_raw = tables["products"].copy()
inventory_raw = tables["inventory"].copy()
variants_raw = tables["variants"].copy()


In [ ]:
# Initial source verification; uniqueness alone does not establish semantic grain.
raw_tables = {
    "customers_raw": (customers_raw, "ID"),
    "orders_raw": (orders_raw, "ID"),
    "products_raw": (products_raw, "ID"),
    "inventory_raw": (inventory_raw, "ID"),
    "variants_raw": (variants_raw, "ID")
}

for name, (df, candidate_key) in raw_tables.items():
    print(
        f"{name}: {len(df):,} rows; "
        f"{df[candidate_key].nunique(dropna=True):,} distinct non-null {candidate_key} values; "
        f"{df[candidate_key].isna().sum():,} nulls"
    )

## 2. Normalize and validate Shopify IDs

Shopify exports globally unique identifiers such as `gid://shopify/Order/...`. Normalized string IDs are easier to join, validate, and eventually store as primary and foreign keys. Working copies protect the untouched raw tables.

In [ ]:
# Normalize Shopify GIDs
def extract_shopify_id(series, entity):
    return (
        series.astype("string")
        .str.extract(fr"^gid://shopify/{entity}/(\d+)$", expand=False)
    )

In [ ]:
# Call extract_shopify_id on orders table
orders_work = orders_raw.copy()

orders_work["order_id"] = extract_shopify_id(orders_work["ID"], "Order")

orders_work["customer_id"] = extract_shopify_id(orders_work["Customer: ID"], "Customer")

orders_work["order_line_id"] = extract_shopify_id(orders_work["Line items: ID"], "LineItem")

orders_work["product_id"] = extract_shopify_id(orders_work["Line items / Product: ID"], "Product")

orders_work["variant_id"] = extract_shopify_id(orders_work["Line items / Variant: ID"], "ProductVariant")

orders_work["fulfillment_id"] = extract_shopify_id(orders_work["Fulfillments: ID"], "Fulfillment")


In [ ]:
# Call extract_shopify_id on customers table
customers_work = customers_raw.copy()

customers_work["customer_id"] = extract_shopify_id(customers_work["ID"], "Customer")

In [ ]:
# Call extract_shopify_id on products table
products_work = products_raw.copy()

products_work["product_id"] = extract_shopify_id(products_work["ID"], "Product")

In [ ]:
# Call extract_shopify_id on inventory table
inventory_work = inventory_raw.copy()

inventory_work["inventory_item_id"] = extract_shopify_id(inventory_work["ID"], "InventoryItem")

inventory_work["variant_id"] = extract_shopify_id(inventory_work["Variant: ID"], "ProductVariant")

inventory_work["inventory_level_id"] = inventory_work["Inventory levels: ID"].str.extract(r"gid://shopify/InventoryLevel/(\d+)\?", expand=False,).astype("string")

inventory_work["location_id"] = extract_shopify_id(inventory_work["Inventory levels / Location: ID"], "Location")

# One current inventory level is identified by its inventory item and level component.
inventory_level_pk = ["inventory_item_id", "inventory_level_id"]

In [ ]:
# Call extract_shopify_id on variants table
variants_work = variants_raw.copy()

variants_work["variant_id"] = extract_shopify_id(variants_work["ID"], "ProductVariant")

variants_work["product_id"] = extract_shopify_id(variants_work["Product: ID"], "Product")

variants_work["inventory_item_id"] = extract_shopify_id(variants_work["Inventory item: ID"], "InventoryItem")

### Validate normalization before modeling

Each check compares a populated raw Shopify GID with its normalized result. A failure means information existed in the source but was lost during extraction. Assertions stop the notebook before invalid IDs can become table keys. The early inventory key is a source-level candidate; its business grain is investigated later.

In [ ]:
# Validate ordinary Shopify GIDs and display all results before failing.
id_checks = [
    ("orders", orders_work, "ID", "order_id"),
    ("orders", orders_work, "Customer: ID", "customer_id"),
    ("orders", orders_work, "Line items: ID", "order_line_id"),
    ("orders", orders_work, "Line items / Product: ID", "product_id"),
    ("orders", orders_work, "Line items / Variant: ID", "variant_id"),
    ("orders", orders_work, "Fulfillments: ID", "fulfillment_id"),

    ("customers", customers_work, "ID", "customer_id"),
    ("products", products_work, "ID", "product_id"),

    ("variants", variants_work, "ID", "variant_id"),
    ("variants", variants_work, "Product: ID", "product_id"),
    ("variants", variants_work, "Inventory item: ID", "inventory_item_id"),

    ("inventory", inventory_work, "ID", "inventory_item_id"),
    ("inventory", inventory_work, "Variant: ID", "variant_id"),
    ("inventory", inventory_work, "Inventory levels / Location: ID", "location_id"),
    ("inventory", inventory_work, "Inventory levels: ID", "inventory_level_id")
]

check_results = []

for table_name, df, raw_column, normalized_column in id_checks:
    failed = df[raw_column].notna() & df[normalized_column].isna() # this identifies rows where Shopify provided a GID but normalization lost it
    failed_count = int(failed.sum())

    check_results.append({
        "table": table_name,
        "raw_column": raw_column,
        "normalized_column": normalized_column,
        "rows": len(df),
        "raw_non_null": int(df[raw_column].notna().sum()),
        "normalized_non_null": int(df[normalized_column].notna().sum()),
        "failed": failed_count,
        "status": "PASS" if failed_count == 0 else "FAIL",
    })

# Dictionaries become a DataFrame, producing an auditable QA table
check_results = pd.DataFrame(check_results)
display(check_results)

failed_checks = check_results.loc[check_results["failed"] > 0]

assert failed_checks.empty, (
    "Some non-null Shopify GIDs failed normalization:\n"
    + failed_checks.to_string(index=False)
)

# Validate the composite primary key for the current Inventory Level snapshot.
inventory_level_null_keys = int(
    inventory_work[inventory_level_pk].isna().any(axis=1).sum()
)
inventory_level_duplicate_keys = int(
    inventory_work.duplicated(inventory_level_pk, keep=False).sum()
)

inventory_level_pk_check = pd.DataFrame([{
    "table": "inventory_levels",
    "primary_key": " + ".join(inventory_level_pk),
    "rows": len(inventory_work),
    "null_key_rows": inventory_level_null_keys,
    "duplicate_key_rows": inventory_level_duplicate_keys,
    "status": (
        "PASS"
        if inventory_level_null_keys == 0 and inventory_level_duplicate_keys == 0
        else "FAIL"
    ),
}])

display(inventory_level_pk_check)

assert inventory_level_null_keys == 0, "Inventory Level PK contains nulls"
assert inventory_level_duplicate_keys == 0, "Inventory Level PK is not unique"

## 3. Separate the nested orders export

The Shopify orders export combines order, order-line, fulfillment, return, and shipping-line data. Its raw row count therefore does not equal its order count. Repeated order-line IDs are investigated before any deduplication to distinguish true conflicts from nested child expansion.

In [ ]:
print("Rows:", len(orders_work))
print("Orders:", orders_work["order_id"].nunique())
print("Order lines:", orders_work["order_line_id"].nunique())
print("Null line IDs:", orders_work["order_line_id"].isna().sum())
print("Duplicate line IDs:", orders_work["order_line_id"].duplicated().sum())

In [ ]:
duplicate_mask = orders_work.duplicated(
    "order_line_id",
    keep=False,
)

duplicate_order_lines = (
    orders_work[duplicate_mask]
    .sort_values(["order_line_id", "order_id"])
)

print(
    f"{len(duplicate_order_lines):,} rows across "
    f"{duplicate_order_lines['order_line_id'].nunique():,} "
    "duplicated order-line IDs"
)

display(duplicate_order_lines.head(50))

In [ ]:
variation_by_line = (
    duplicate_order_lines
    .groupby("order_line_id")
    .nunique(dropna=False)
)

varying_columns = (
    variation_by_line.gt(1)
    .sum()
    .sort_values(ascending=False)
)

display(
    varying_columns[
        varying_columns > 0
    ].to_frame("duplicate_groups_with_variation")
)

### 3.1 Construct order lines

The investigation shows that repeated line IDs vary only because lower-level fulfillment, return, or shipping records expand the export. Retaining only line-level attributes produces a table whose contract is **one row per `order_line_id`**.

In [ ]:
order_line_columns = [
    "order_line_id",
    "order_id",
    "product_id",
    "variant_id",
    "Line items: Quantity",
    "Line items: Unfulfilled quantity",
    "Line items: Fulfillable quantity",
    "Line items: Title",
    "Line items: Variant title",
    "Line items: Shop original unit price",
    "Line items: Shop original unit price currency",
    "Line items: Shop original total",
    "Line items: Shop original total currency",
    "Line items: Shop discounted total",
    "Line items: Shop discounted total currency",
    "Line items: Shop total discount",
    "Line items: Shop total discount currency",
    "Line items: Taxable",
    "Line items: Requires shipping",
]

In [ ]:
# Remove identical rows and validate
order_lines_clean = (
    orders_work[order_line_columns]
    .drop_duplicates()
    .copy()
)

assert order_lines_clean["order_line_id"].notna().all(), (
    "Order-line IDs contain nulls"
)

assert order_lines_clean["order_line_id"].is_unique, (
    "Some order lines have conflicting retained attributes"
)

### 3.2 Construct fulfillments

Fulfillments are optional child records of orders. Rows without a fulfillment ID are excluded; the resulting contract is **one row per `fulfillment_id`**, with `order_id` linking each fulfillment to its parent order.

In [ ]:
fulfillments_clean = (
    orders_work[
        ["fulfillment_id", "order_id"]
    ]
    .dropna(subset=["fulfillment_id"])
    .drop_duplicates()
    .copy()
)

In [ ]:
print("Rows:", len(fulfillments_clean))
print("Null fulfillment IDs:", fulfillments_clean["fulfillment_id"].isna().sum())
print("Duplicate fulfillment IDs:", fulfillments_clean["fulfillment_id"].duplicated().sum())
print("Null order IDs:", fulfillments_clean["order_id"].isna().sum())

assert fulfillments_clean["fulfillment_id"].notna().all()
assert fulfillments_clean["fulfillment_id"].is_unique
assert fulfillments_clean["order_id"].notna().all()

### 3.3 Construct orders

Order-level timestamps, statuses, totals, and shipping geography are retained once per order. Shipping geography belongs here because it describes where a specific sale was shipped, which is more appropriate for regional sales analysis than a customer's current saved address.

In [ ]:
order_columns = [
    "order_id",
    "customer_id",
    "Name",
    "Created at",
    "Updated at",
    "Processed at",
    "Cancelled at",
    "Shipping address: City",
    "Shipping address: Province", 
    "Shipping address: Country",
    "Display financial status",
    "Display fulfillment status",
    "Payment gateways",
    "Currency code",
    "Presentment currency",
    "Shop subtotal price",
    "Shop subtotal price currency",
    "Shop total shipping price",
    "Shop total shipping price currency",
    "Shop total tax",
    "Shop total tax currency",
    "Shop total discounts",
    "Shop total discounts currency",
    "Shop total price",
    "Shop total price currency",
    "Test",
]

In [ ]:
# Create new orders table using order level fields from order_columns
orders_clean = (
    orders_work[order_columns]
    .drop_duplicates()
    .copy()
)

# test row counts
print("Orders:", len(orders_clean))
print("Null order IDs:", orders_clean["order_id"].isna().sum())
print("Duplicate order IDs:", orders_clean["order_id"].duplicated().sum())

# lock in contract
assert orders_clean["order_id"].notna().all()
assert orders_clean["order_id"].is_unique


### Validate order relationships

Every order line and fulfillment must reference an order in `orders_clean`. An orphan count above zero would indicate that a child record lost its parent during cleaning or that the source extracts are incomplete.

In [ ]:
print(
    "Order-line orphan rows:",
    (~order_lines_clean["order_id"].isin(orders_clean["order_id"])).sum()
)

print(
    "Fulfillment orphan rows:",
    (~fulfillments_clean["order_id"].isin(orders_clean["order_id"])).sum()
)

assert order_lines_clean["order_id"].isin(
    orders_clean["order_id"]
).all()

assert fulfillments_clean["order_id"].isin(
    orders_clean["order_id"]
).all()

## 4. Construct customers

The customer table keeps customer-level identity, lifecycle, and segmentation attributes at **one row per `customer_id`**. Saved-address fields are excluded because they can create multiple rows per customer and do not reliably represent the shipping destination of a particular order.

> **Privacy:** Names, email addresses, and phone numbers are personally identifiable information. They may be needed internally, but they must not be displayed in the public portfolio dashboard or committed as customer-level data.

In [ ]:
for column in customers_work.columns:
    print(column)

In [ ]:
customer_columns = [
    "customer_id", 
    "First name",
    "Last name", 
    "Lifetime duration", 
    "Email", 
    "Phone", 
    "Created at",
    "Number of orders",
    "Amount spent", 
    "Updated at",
    "State", 
    "Tags"
]

In [ ]:
# Create customers clean table using customer level columns from customers_work; customer_columns variable
customers_clean = (
    customers_work[customer_columns]
    .drop_duplicates()
    .copy()
)

# Test customer row count; whether one customer_id gives one customer row
print("Customers:", len(customers_clean))
print("Null customer_id count:", customers_clean["customer_id"].isna().sum())
print("Duplicate customer_id count:", customers_clean["customer_id"].duplicated().sum())

assert customers_clean["customer_id"].notna().all()
assert customers_clean["customer_id"].is_unique

### Validate orders to customers

A null customer ID is treated separately from an unmatched non-null ID. Nulls can represent guest, deleted, or anonymized customers; a populated ID missing from `customers_clean` would indicate a referential-integrity problem.

In [ ]:
# Validate relationship between orders_clean and customers_clean
order_customer_ids = orders_clean["customer_id"]

print("Orders with null customer_id:", orders_clean["customer_id"].isna().sum())
print("Orders with unmatched customer_id:", (~order_customer_ids.dropna().isin(customers_clean["customer_id"])).sum())

assert order_customer_ids.dropna().isin(customers_clean["customer_id"]).all()

## 5. Construct products

Product-level catalog attributes are retained at **one row per `product_id`**. Current inventory totals are excluded because inventory is modeled separately at the variant, inventory-item, and location levels.

In [ ]:
for column in products_work.columns:
    print(column)

In [ ]:
product_columns = [
    "product_id",
    "Title",
    "Vendor",
    "Product type",
    "Status",
    "Tags",
    "Created at",
    "Updated at",
    "Published at"    
]

In [ ]:
products_clean = (
    products_work[product_columns]
    .drop_duplicates()
    .copy()
)

print("Product row count:", len(products_clean))
print("Products with null product_id:", products_clean["product_id"].isna().sum())
print("Duplicated product_id count:", products_clean["product_id"].duplicated().sum())

assert products_clean["product_id"].notna().all()
assert products_clean["product_id"].is_unique

### Validate order lines to products

Missing product IDs are reported separately because historical lines may outlive deleted catalog records. Every populated `product_id`, however, must match a parent row in `products_clean`.

In [ ]:
order_line_product_ids = order_lines_clean["product_id"] # Which product, if any, each order line is connected to

# Identify nulls -- then check if non-null product_id in order_lines_clean match products_clean product_id
print("Null product_id in order_lines_clean count:", order_lines_clean["product_id"].isna().sum())
print("Unmatched product_id count:", (~order_line_product_ids.dropna().isin(products_clean["product_id"])).sum()) 

assert order_line_product_ids.dropna().isin(products_clean["product_id"]).all()

## 6. Construct variants

Variants represent sellable product options and connect products to inventory items. The contract is **one row per `variant_id`**. Nested selected-option fields are excluded because they expand a single variant across multiple source rows.

In [ ]:
for column in variants_work.columns:
    print(column)

In [ ]:
variant_columns = [
    "variant_id",
    "product_id",
    "inventory_item_id",
    "Title",
    "Created at",
    "Updated at",
    "SKU",
    "Price",
    "Inventory item: Unit cost",
    "Compare-at price",
    "Taxable",
    "Available for sale"
]

In [ ]:
variants_clean = (
    variants_work[variant_columns]
    .drop_duplicates()
    .copy()
)

print("Variants count:", len(variants_clean))
print("Null variant_id count:", variants_clean["variant_id"].isna().sum())
print("Duplicated variant_id count:", variants_clean["variant_id"].duplicated().sum())

assert variants_clean["variant_id"].notna().all()
assert variants_clean["variant_id"].is_unique

### Validate variants to products

Every populated variant `product_id` must match a parent product. Null and unmatched references are counted separately so optionality is not confused with broken integrity.

In [ ]:
variant_product_ids = variants_clean["product_id"]

print("Null product_id in variants_clean count:", variants_clean["product_id"].isna().sum())
print("Unmatched product_id count:", (~variant_product_ids.dropna().isin(products_clean["product_id"])).sum())

assert variant_product_ids.dropna().isin(products_clean["product_id"]).all()

### Validate order lines to variants

Historical order lines may have no variant reference, but every populated `variant_id` must match `variants_clean`. This preserves historical sales while still enforcing integrity where Shopify provides a key.

In [ ]:
order_line_variant_ids = order_lines_clean["variant_id"]

print("Null variant_id in order_line_clean count:", order_lines_clean["variant_id"].isna().sum())
print("Unmatched variant_id count:", (~order_line_variant_ids.dropna().isin(variants_clean["variant_id"])).sum())

assert order_line_variant_ids.dropna().isin(variants_clean["variant_id"]).all()

## 7. Investigate inventory

The inventory export combines stable inventory-item attributes with location-specific quantity states. It is a **current snapshot**, so it supports current stock analysis but cannot reconstruct inventory at the time of historical orders. The grain is validated before creating either inventory table.

In [ ]:
inventory_work.columns

In [ ]:
inventory_level_columns = [
    "inventory_item_id",
    "inventory_level_id",
    "location_id",
    "Inventory levels / Location: Name",
    "Inventory levels / Quantities: Available quantity",
    "Inventory levels / Quantities: On hand quantity",
    "Inventory levels / Quantities: Committed quantity",
    "Inventory levels / Quantities: Incoming quantity",
    "Inventory levels / Quantities: Reserved quantity",
    "Inventory levels / Quantities: Safety stock quantity",
    "Inventory levels / Quantities: Available quantity updated at"
]

### 7.1 Establish the raw inventory grain

Compare rows with distinct inventory items, level components, and locations. More rows than items suggests that items are repeated across a lower-level relationship rather than duplicated accidentally.

In [ ]:
print("Rows:", len(inventory_work))

print("Distinct inventory items:", inventory_work["inventory_item_id"].nunique())

print("Distinct inventory levels:", inventory_work["inventory_level_id"].nunique())

print("Distinct inventory locations:", inventory_work["location_id"].nunique())

### 7.2 Interpret level and location identifiers

The extracted level component and Shopify location ID are not assumed to be identical. Their distinct mapping is inspected to determine whether they describe the same three business locations through different source identifiers.

In [ ]:
level_location_match = (
    inventory_work["inventory_level_id"] 
    == inventory_work["location_id"]
)

print("Level ID always equals location ID:", level_location_match.all())

In [ ]:
# Show the distinct technical level-to-location relationships.
level_location_map = (
    inventory_work[
        [
            "inventory_level_id",
            "location_id",
            "Inventory levels / Location: Name",
        ]
    ]
    .drop_duplicates()
    .sort_values("inventory_level_id")
)

display(level_location_map)

In [ ]:
# Confirm that each technical level maps to one location and vice versa.
locations_per_level = (
    level_location_map
    .groupby("inventory_level_id")["location_id"]
    .nunique()
)

levels_per_location = (
    level_location_map
    .groupby("location_id")["inventory_level_id"]
    .nunique()
)

print("Levels mapped to multiple locations:", locations_per_level.gt(1).sum())
print("Locations mapped to multiple levels:", levels_per_location.gt(1).sum())

assert locations_per_level.eq(1).all()
assert levels_per_location.eq(1).all()

### 7.3 Validate the item-location grain

Count locations per inventory item, then test `inventory_item_id + location_id` as the business key. A valid key must be complete and unique. This proves that repeated inventory-item IDs represent legitimate location assignments.

In [ ]:
locations_per_item = (
    inventory_work
    .groupby("inventory_item_id")["location_id"]
    .nunique()
)

print(
    "Inventory items at multiple locations:",
    locations_per_item.gt(1).sum()
)

display(
    locations_per_item
    .sort_values(ascending=False)
    .head(10)
)

In [ ]:
inventory_level_key = [
    "inventory_item_id",
    "location_id",
]

print(
    "Rows with null key values:",
    inventory_work[inventory_level_key]
    .isna()
    .any(axis=1)
    .sum()
)

print(
    "Duplicate inventory-level keys:",
    inventory_work
    .duplicated(inventory_level_key)
    .sum()
)

assert inventory_work[inventory_level_key].notna().all().all()
assert not inventory_work.duplicated(inventory_level_key).any()

In [ ]:
duplicate_inventory_levels = (
    inventory_work[
        inventory_work.duplicated(
            inventory_level_key,
            keep=False,
        )
    ]
    .sort_values(inventory_level_key)
)

display(duplicate_inventory_levels.head(20))

### 7.4 Validate inventory-item attributes

Fields retained for `inventory_items_clean` must remain identical when the same item appears at multiple locations. Any variation would mean the proposed columns do not share a one-row-per-item grain and must be investigated before deduplication.

In [ ]:
inventory_item_columns = [
    "inventory_item_id",
    "variant_id",
    "Variant: SKU",
    "Tracked",
    "Unit cost",
    "Unit cost currency",
    "Requires shipping",
]

In [ ]:
# Find inventory items whose retained attributes conflict across locations.
inventory_item_attributes = [
    column
    for column in inventory_item_columns
    if column != "inventory_item_id"
]

inventory_item_variation = (
    inventory_work
    .groupby("inventory_item_id")[inventory_item_attributes]
    .nunique(dropna=False)
)

conflicting_inventory_items = (
    inventory_item_variation.gt(1).any(axis=1)
)

print("Inventory items with conflicting attributes:", conflicting_inventory_items.sum())
display(inventory_item_variation[conflicting_inventory_items].head(20))

assert not conflicting_inventory_items.any()

## Ready to construct the inventory clean tables

- `inventory_items_clean`: one row per `inventory_item_id`
- `inventory_levels_clean`: one row per `inventory_item_id` and `location_id`

The next cells will construct these tables only after the grain and retained attributes pass validation.

### 7.5 Construct inventory items

Location-specific repetitions can now be removed because the retained item attributes were proven stable. The resulting table contract is **one row per `inventory_item_id`**.

In [ ]:
inventory_items_clean = (
    inventory_work[inventory_item_columns]
    .drop_duplicates()
    .copy()
)

print("Inventory items:", len(inventory_items_clean))
print("Null inventory_item_id count:", inventory_items_clean["inventory_item_id"].isna().sum())
print("Duplicate inventory_item_id count:", inventory_items_clean["inventory_item_id"].duplicated().sum())

assert inventory_items_clean["inventory_item_id"].notna().all()
assert inventory_items_clean["inventory_item_id"].is_unique

### Validate variants to inventory items

Shopify models a variant and its inventory item as a one-to-one relationship. Both directions are checked so that every populated reference resolves and neither table assigns the same inventory object to multiple records.

In [ ]:
inventory_variant_ids = inventory_items_clean["variant_id"]

print("Inventory items with null variant_id:", inventory_variant_ids.isna().sum())
print(
    "Inventory items with unmatched variant_id:",
    (~inventory_variant_ids.dropna().isin(variants_clean["variant_id"])).sum(),
)
print("Duplicate variant_id assignments:", inventory_variant_ids.dropna().duplicated().sum())

assert inventory_variant_ids.dropna().isin(variants_clean["variant_id"]).all()
assert inventory_variant_ids.dropna().is_unique

In [ ]:
variant_inventory_item_ids = variants_clean["inventory_item_id"]

print("Variants with null inventory_item_id:", variant_inventory_item_ids.isna().sum())
print(
    "Variants with unmatched inventory_item_id:",
    (~variant_inventory_item_ids.dropna().isin(inventory_items_clean["inventory_item_id"])).sum(),
)
print("Duplicate inventory_item_id assignments:", variant_inventory_item_ids.dropna().duplicated().sum())

assert variant_inventory_item_ids.dropna().isin(inventory_items_clean["inventory_item_id"]).all()
assert variant_inventory_item_ids.dropna().is_unique

### 7.6 Construct inventory levels

Each row records the current quantity states of one inventory item at one Shopify location. `inventory_level_id` is preserved for source traceability, while `inventory_item_id + location_id` expresses the business grain.

In [ ]:
inventory_levels_clean = (
    inventory_work[inventory_level_columns]
    .copy()
)

print("Inventory levels:", len(inventory_levels_clean))
print(
    "Rows with null inventory-level keys:",
    inventory_levels_clean[inventory_level_key].isna().any(axis=1).sum(),
)
print(
    "Duplicate inventory-level keys:",
    inventory_levels_clean.duplicated(inventory_level_key).sum(),
)

assert inventory_levels_clean[inventory_level_key].notna().all().all()
assert not inventory_levels_clean.duplicated(inventory_level_key).any()

### Validate inventory levels to inventory items

Every location-level inventory record must reference an item in `inventory_items_clean`. An orphan would indicate that the parent item was lost during cleaning.

In [ ]:
inventory_level_item_ids = inventory_levels_clean["inventory_item_id"]

print(
    "Inventory-level orphan rows:",
    (~inventory_level_item_ids.isin(inventory_items_clean["inventory_item_id"])).sum(),
)

assert inventory_level_item_ids.isin(
    inventory_items_clean["inventory_item_id"]
).all()

## Inventory cleaning complete

The inventory model now separates stable item attributes from location-specific quantity states:

- `variants_clean` identifies the sellable catalog option.
- `inventory_items_clean` identifies its stock-tracked inventory object.
- `inventory_levels_clean` records the current state of that object at each Shopify location.

> This export is a current snapshot. It supports current availability and location analysis, but not historical inventory at the time an earlier order was placed.

## 8. Validate data-types and establish contracts

In [ ]:
orders_clean.dtypes

Created at, updated at, processed at, cancelled at all need to be converted to datetime


In [ ]:
order_timestamp_columns = [
    "Created at", 
    "Updated at", 
    "Processed at", 
    "Cancelled at"
]

for column in order_timestamp_columns:
    original = orders_clean[column].copy()
    converted = pd.to_datetime(original, errors="coerce", utc=True)

    failed = original.notna() & converted.isna()
    failed_count = int(failed.sum())
    print(column, "conversion failures", failed_count) # if all pass, this proves that every populated source timestamp parsed successfully

    assert failed_count == 0, f"{column} has {failed_count} conversion failures" 

    orders_clean[column] = converted

In [ ]:
orders_clean[order_timestamp_columns].dtypes

In [ ]:
# Validate date ranges are within expected ranges
orders_clean[order_timestamp_columns].agg(["min","max"])

In [ ]:
# Validate that updated at date does not come before created at date
updated_before_created_count = (orders_clean["Updated at"] < orders_clean["Created at"]).sum()

print("Orders updated before creation:", updated_before_created_count)

assert updated_before_created_count == 0, f"There are {updated_before_created_count} rows updated before creation"

In [ ]:
# Validate Cancelled at date does not come before Created at date
cancelled_before_created_count = (orders_clean["Cancelled at"] < orders_clean["Created at"]).sum()

print("Orders cancelled before creation:", cancelled_before_created_count)

assert cancelled_before_created_count == 0, f"There are {cancelled_before_created_count} rows cancelled before creation"

In [ ]:
order_lines_clean.dtypes

In [ ]:
quantity_check_columns = [
    "Line items: Quantity", 
    "Line items: Unfulfilled quantity", 
    "Line items: Fulfillable quantity"
]

order_lines_clean[quantity_check_columns].agg(["min", "max"])

In [ ]:
zero_or_less_quantity_count = (order_lines_clean["Line items: Quantity"] <= 0).sum()

print("Order lines with zero or less quantity:", zero_or_less_quantity_count)

assert zero_or_less_quantity_count == 0, f"There are {zero_or_less_quantity_count} order lines less than, or equal to, 0"

In [ ]:
remaining_quantity_columns = [
    "Line items: Unfulfilled quantity",
    "Line items: Fulfillable quantity"
]

for column in remaining_quantity_columns:
    less_than_zero = (order_lines_clean[column] < 0).sum()

    print(f"{column} column has", less_than_zero, "rows with count less than zero")

    assert less_than_zero == 0, f"There are {less_than_zero} order lines less than zero"

In [ ]:
# Unfulfilled order line quantity cannot be greater than quantity originally ordered

unfulfilled_exceeds_quantity_count = (order_lines_clean["Line items: Unfulfilled quantity"] > order_lines_clean["Line items: Quantity"]).sum()

print("The number of order lines where unfulfilled quantity is greater than quantity is", unfulfilled_exceeds_quantity_count)

assert unfulfilled_exceeds_quantity_count == 0, f"There are {unfulfilled_exceeds_quantity_count} order lines where unfulfilled quantity exceeds ordered quantity"

In [ ]:
# Fulfillable quantity cannot exceed the unfulfilled quantity on an order line.
fulfillable_exceeds_unfulfilled_count = (
    order_lines_clean["Line items: Fulfillable quantity"]
    > order_lines_clean["Line items: Unfulfilled quantity"]
).sum()

print(
    "Order lines where fulfillable quantity exceeds unfulfilled quantity:",
    fulfillable_exceeds_unfulfilled_count,
)

assert fulfillable_exceeds_unfulfilled_count == 0, (
    f"There are {fulfillable_exceeds_unfulfilled_count} order lines where "
    "fulfillable quantity exceeds unfulfilled quantity"
)